In [1]:
# ============================================================
# ĐỌC 4 FILE ĐÃ XỬ LÝ VÀ KIỂM TRA SHAPE
# ============================================================
# Mục đích: xem kích thước từng bảng trước khi merge,
# để biết bảng nào thiếu bệnh nhân, bảng nào đủ.

import pandas as pd
import numpy as np

# Đường dẫn (tính từ data_processing/)
PATIENT_PATH = "../data/processed/patient_clean.csv"
GCS_PATH     = "../data/processed/gcs_clean.csv"
VITAL_PATH   = "../data/processed/vital_clean.csv"
LAB_PATH     = "../data/processed/lab_clean.csv"

# Đọc 4 file
df_patient = pd.read_csv(PATIENT_PATH)
df_gcs     = pd.read_csv(GCS_PATH)
df_vital   = pd.read_csv(VITAL_PATH)
df_lab     = pd.read_csv(LAB_PATH)

In [2]:
# In shape
print("=" * 60)
print("KÍCH THƯỚC 4 BẢNG ĐÃ XỬ LÝ")
print("=" * 60)
print(f"patient_clean : {df_patient.shape[0]:>5,} dòng × {df_patient.shape[1]:>2} cột")
print(f"gcs_clean     : {df_gcs.shape[0]:>5,} dòng × {df_gcs.shape[1]:>2} cột")
print(f"vital_clean   : {df_vital.shape[0]:>5,} dòng × {df_vital.shape[1]:>2} cột")
print(f"lab_clean     : {df_lab.shape[0]:>5,} dòng × {df_lab.shape[1]:>2} cột")

# In cột của từng bảng
print("\n" + "=" * 60)
print("CỘT CỦA TỪNG BẢNG")
print("=" * 60)
print(f"\npatient_clean:\n  {list(df_patient.columns)}")
print(f"\ngcs_clean:\n  {list(df_gcs.columns)}")
print(f"\nvital_clean:\n  {list(df_vital.columns)}")
print(f"\nlab_clean:\n  {list(df_lab.columns)}")

# Kiểm tra khóa chính duy nhất ở từng bảng
print("\n" + "=" * 60)
print("KIỂM TRA KHÓA CHÍNH")
print("=" * 60)
for name, df in [("patient", df_patient), ("gcs", df_gcs),
                 ("vital", df_vital), ("lab", df_lab)]:
    n_dup = df["patientunitstayid"].duplicated().sum()
    print(f"{name:10s}: trùng lặp = {n_dup} "
          f"{'✅' if n_dup == 0 else '⚠️'}")

KÍCH THƯỚC 4 BẢNG ĐÃ XỬ LÝ
patient_clean : 2,326 dòng ×  5 cột
gcs_clean     : 2,074 dòng ×  2 cột
vital_clean   : 2,240 dòng × 10 cột
lab_clean     : 2,265 dòng × 11 cột

CỘT CỦA TỪNG BẢNG

patient_clean:
  ['patientunitstayid', 'age', 'male_sex', 'bmi', 'mortality']

gcs_clean:
  ['patientunitstayid', 'gcs']

vital_clean:
  ['patientunitstayid', 'heartrate_mean', 'heartrate_min', 'heartrate_max', 'systemicmean_max', 'sao2_mean', 'sao2_min', 'respiration_mean', 'respiration_max', 'temperature_mean']

lab_clean:
  ['patientunitstayid', 'creatinine_max', 'bun_mean', 'lactate_mean', 'lactate_max', 'wbc_mean', 'hemoglobin_mean', 'platelets_mean', 'bilirubin_mean', 'glucose_mean', 'glucose_max']

KIỂM TRA KHÓA CHÍNH
patient   : trùng lặp = 0 ✅
gcs       : trùng lặp = 0 ✅
vital     : trùng lặp = 0 ✅
lab       : trùng lặp = 0 ✅


In [3]:
# ============================================================
# KIỂM TRA MỨC ĐỘ PHỦ SÓNG
# ============================================================
# Bao nhiêu bệnh nhân trong patient_clean có mặt ở từng bảng?

print("=" * 60)
print("MỨC ĐỘ PHỦ SÓNG (COVERAGE)")
print("=" * 60)

valid_ids = set(df_patient["patientunitstayid"])
print(f"\nTổng cohort (patient_clean): {len(valid_ids):,} bệnh nhân\n")

for name, df in [("gcs", df_gcs), ("vital", df_vital), ("lab", df_lab)]:
    ids = set(df["patientunitstayid"])
    in_cohort = ids & valid_ids
    pct = len(in_cohort) / len(valid_ids) * 100
    print(f"{name:10s}: {len(in_cohort):>5,} / {len(valid_ids):,} "
          f"({pct:>5.1f}%) bệnh nhân cohort có dữ liệu")

MỨC ĐỘ PHỦ SÓNG (COVERAGE)

Tổng cohort (patient_clean): 2,326 bệnh nhân

gcs       : 2,074 / 2,326 ( 89.2%) bệnh nhân cohort có dữ liệu
vital     : 2,240 / 2,326 ( 96.3%) bệnh nhân cohort có dữ liệu
lab       : 2,161 / 2,326 ( 92.9%) bệnh nhân cohort có dữ liệu


In [4]:
# ============================================================
# MERGE 4 BẢNG — LEFT JOIN TỪ PATIENT
# ============================================================
# Dùng how='left' từ patient_clean để đảm bảo ĐỦ 2,326 bệnh nhân.
# Bệnh nhân nào thiếu bảng nào -> cột đó = NaN.
# Thứ tự merge không quan trọng vì tất cả đều left join từ patient.

print("=" * 60)
print("MERGE 4 BẢNG")
print("=" * 60)

df_ai_ready = df_patient.copy()
print(f"Bắt đầu với patient_clean: {df_ai_ready.shape}")

for name, df in [("gcs", df_gcs), ("vital", df_vital), ("lab", df_lab)]:
    n_before = len(df_ai_ready)
    df_ai_ready = df_ai_ready.merge(df, on="patientunitstayid", how="left")
    print(f"Sau merge {name:6s}: {df_ai_ready.shape} "
          f"(số dòng: {n_before} -> {len(df_ai_ready)})")

# ---------- Kiểm tra sau merge ----------
print("\n" + "=" * 60)
print("KIỂM TRA SAU MERGE")
print("=" * 60)

# Kích thước
print(f"Kích thước cuối: {df_ai_ready.shape}")
assert len(df_ai_ready) == len(df_patient), \
    f"Số dòng không khớp: {len(df_ai_ready)} vs {len(df_patient)}"
print(f"✅ Số dòng khớp với patient_clean ({len(df_patient):,})")

# Khóa chính duy nhất
n_dup = df_ai_ready["patientunitstayid"].duplicated().sum()
print(f"{'✅' if n_dup == 0 else '⚠️'} patientunitstayid trùng lặp: {n_dup}")
assert n_dup == 0

# Nhãn mortality không NaN
n_nan_label = df_ai_ready["mortality"].isna().sum()
print(f"{'✅' if n_nan_label == 0 else '⚠️'} mortality NaN: {n_nan_label}")

print("\n5 dòng đầu tiên:")
display(df_ai_ready.head())

MERGE 4 BẢNG
Bắt đầu với patient_clean: (2326, 5)
Sau merge gcs   : (2326, 6) (số dòng: 2326 -> 2326)
Sau merge vital : (2326, 15) (số dòng: 2326 -> 2326)
Sau merge lab   : (2326, 25) (số dòng: 2326 -> 2326)

KIỂM TRA SAU MERGE
Kích thước cuối: (2326, 25)
✅ Số dòng khớp với patient_clean (2,326)
✅ patientunitstayid trùng lặp: 0
✅ mortality NaN: 0

5 dòng đầu tiên:


,patientunitstayid,age,male_sex,bmi,mortality,gcs,heartrate_mean,heartrate_min,heartrate_max,systemicmean_max,...,creatinine_max,bun_mean,lactate_mean,lactate_max,wbc_mean,hemoglobin_mean,platelets_mean,bilirubin_mean,glucose_mean,glucose_max
0,141764,87.0,0.0,NaN,0,NaN,106.652174,90.0,138.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,141765,87.0,0.0,18.745276,0,15.0,83.190972,72.0,116.0,NaN,...,1.04,28.0,NaN,NaN,10.2,12.3,191.0,NaN,75.0,89.0
2,143870,76.0,1.0,27.788734,0,15.0,45.449367,40.0,55.0,79.0,...,1.14,13.5,NaN,NaN,11.7,12.3,152.0,NaN,131.5,140.0
3,144815,34.0,0.0,20.217741,0,15.0,83.800000,68.0,118.0,NaN,...,0.63,6.0,NaN,NaN,7.9,12.3,319.0,0.5,82.0,82.0
4,145427,61.0,1.0,29.007201,0,15.0,65.457983,49.0,85.0,NaN,...,1.05,14.0,NaN,NaN,21.1,13.3,196.0,NaN,128.5,139.0


In [5]:
# ============================================================
# CELL 3: TỶ LỆ NaN, THỐNG KÊ, LƯU FILE
# ============================================================

feature_cols = [c for c in df_ai_ready.columns
                if c not in ["patientunitstayid", "mortality"]]

# ---------- Tỷ lệ NaN ----------
print("=" * 60)
print("TỶ LỆ NaN CỦA TỪNG ĐẶC TRƯNG")
print("=" * 60)

print(f"\n{'Đặc trưng':22s} {'NaN':>6s} {'Tỷ lệ':>8s}  Đánh giá")
print("-" * 60)
for col in feature_cols:
    n_miss = df_ai_ready[col].isna().sum()
    pct = n_miss / len(df_ai_ready) * 100
    if pct < 10:
        status = "✅ Tốt"
    elif pct < 50:
        status = "ℹ️ Chấp nhận"
    else:
        status = "⚠️ Rất thưa"
    print(f"{col:22s} {n_miss:>6,} {pct:>7.2f}%  {status}")

# ---------- Thống kê mô tả ----------
print("\n" + "=" * 60)
print("THỐNG KÊ MÔ TẢ")
print("=" * 60)
print(df_ai_ready[feature_cols + ["mortality"]].describe().round(2).to_string())

# ---------- Mất cân bằng nhãn ----------
print("\n" + "=" * 60)
print("MẤT CÂN BẰNG NHÃN")
print("=" * 60)

vc = df_ai_ready["mortality"].value_counts()
vc_pct = df_ai_ready["mortality"].value_counts(normalize=True) * 100
print(f"  Alive   (0): {vc.get(0, 0):,} ({vc_pct.get(0, 0):.2f}%)")
print(f"  Expired (1): {vc.get(1, 0):,} ({vc_pct.get(1, 0):.2f}%)")
spw = vc.get(0, 1) / vc.get(1, 1)
print(f"  scale_pos_weight gợi ý: {spw:.2f}")



TỶ LỆ NaN CỦA TỪNG ĐẶC TRƯNG

Đặc trưng                 NaN    Tỷ lệ  Đánh giá
------------------------------------------------------------
age                         0    0.00%  ✅ Tốt
male_sex                    2    0.09%  ✅ Tốt
bmi                       169    7.27%  ✅ Tốt
gcs                       303   13.03%  ℹ️ Chấp nhận
heartrate_mean             97    4.17%  ✅ Tốt
heartrate_min              97    4.17%  ✅ Tốt
heartrate_max              97    4.17%  ✅ Tốt
systemicmean_max        1,941   83.45%  ⚠️ Rất thưa
sao2_mean                 122    5.25%  ✅ Tốt
sao2_min                  122    5.25%  ✅ Tốt
respiration_mean          291   12.51%  ℹ️ Chấp nhận
respiration_max           291   12.51%  ℹ️ Chấp nhận
temperature_mean        2,181   93.77%  ⚠️ Rất thưa
creatinine_max            316   13.59%  ℹ️ Chấp nhận
bun_mean                  319   13.71%  ℹ️ Chấp nhận
lactate_mean            1,870   80.40%  ⚠️ Rất thưa
lactate_max             1,870   80.40%  ⚠️ Rất thưa
wbc_mean           

In [6]:
# ---------- Lưu file ----------
import os
OUTPUT_PATH = "../data/processed/ai_ready.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df_ai_ready.to_csv(OUTPUT_PATH, index=False)

print("\n" + "=" * 60)
print(f"✅ Đã lưu: {OUTPUT_PATH}")
print(f"   Kích thước cuối: {df_ai_ready.shape}")
print("=" * 60)


✅ Đã lưu: ../data/processed/ai_ready.csv
   Kích thước cuối: (2326, 25)
